In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
import numpy as np
import os
import torch
import pickle
import config
import json

In [3]:
from sentence_transformers import CrossEncoder,InputExample
from torch.utils.data import DataLoader
from src.metric import model_evaluation


In [4]:
path=config.CLEANED_DATA_DIR

In [5]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    


In [6]:
BATCH_SIZE=config.BASELINE_BATCH_SIZE

In [7]:

cross_train_examples=[
    InputExample(texts=[j,r],label=float(config.label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

print(f"total training example:{len(cross_train_examples)}")

total training example:6240


In [8]:
cross_train_dataloader=DataLoader(cross_train_examples,shuffle=True,batch_size=BATCH_SIZE)

In [9]:
cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',num_labels=1,device=config.device)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [10]:
test_pairs=list(zip(test_df['job_description_text'],test_df['resume_text']))

scores=cross_encoder_model.predict(test_pairs,batch_size=BATCH_SIZE,show_progress_bar=False)

metrics=model_evaluation(scores,test_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])


Spearman: 0.15692132526881591
Top-3 Accuracy: 0.9285714285714286
NDCG: 0.5601103442081681
MAP: 0.6565044443862946
MRR: 0.7311475409836066


In [11]:
results_dir=config.RESULTS_DIR
os.makedirs(results_dir,exist_ok=True)

with open(os.path.join(results_dir,'cross_encoder_pretrained.json'),'w') as f:
    json.dump(metrics,f)

In [12]:
model_save_path=os.path.join(config.BASELINE_MODEL_DIR,'cross_encoder')
os.makedirs(config.BASELINE_MODEL_DIR,exist_ok=True)

In [13]:
epochs=4
best_score=float('-inf')
min_delta=0.01
patience=2
count=0

for epoch in range(1,epochs+1):
    print(f"Epoch: {epoch}----------")

    cross_encoder_model.fit(train_dataloader=cross_train_dataloader,epochs=1,
                             warmup_steps=int(len(cross_train_dataloader) * epochs * 0.1),
                            show_progress_bar=True)
    
    val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

    scores=cross_encoder_model.predict(val_pairs,batch_size=BATCH_SIZE,show_progress_bar=False)
    
    metrics=model_evaluation(scores,val_df,'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])    

    final_score = 0.6*metrics['ndcg_val']+0.3*metrics['map_score']+0.1*metrics['mrr_score']
    
    if final_score>best_score + min_delta:
        best_score=final_score
        cross_encoder_model.save(model_save_path)
        count=0
    else:
        count+=1
        
    if count==patience:
        print("Early Stopping")
        break
    


Epoch: 1----------


Step,Training Loss


NDCG: 0.7161667965853731
MAP: 0.7938697560625837


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 2----------


...

NDCG: 0.7328552168432514
MAP: 0.8010732338650758


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 4----------


Step,Training Loss


NDCG: 0.741245126930484
MAP: 0.8084476748866246


In [14]:
cross_encoder_model=CrossEncoder(model_save_path,device=config.device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [15]:
val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

scores=cross_encoder_model.predict(val_pairs,batch_size=64,show_progress_bar=False)


In [16]:
metrics=model_evaluation(scores,val_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.4703216097756826
Top-3 Accuracy: 1.0
NDCG: 0.7328552168432514
MAP: 0.8010732338650758
MRR: 0.8723958333333334


In [17]:
ranked_result=[]
eval_df=val_df.copy()
eval_df['score']=scores

In [18]:
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")


Score spread within groups:
Mean spread: 3.492
% groups with spread < 0.1:0.000


In [19]:
for jd,group in eval_df.groupby('job_description_text'):
    ranked_group=group.sort_values("score",ascending=False)
    ranked_result.append(ranked_group)

final_rank_df=pd.concat(ranked_result)

In [20]:
i=0
for jd,group in final_rank_df.groupby('job_description_text'):
    if(len(group)>2 and len(group)<10):
        print("Job Description:\n",jd[:300])
        print(group[['label','score']])
        i+=1
        if i==3:
            break

Job Description:
 About Chamberlain Group:
Chamberlain Group is a global leader in access solutions. Our leading brands like LiftMaster, Chamberlain, Merlin and Grifco are found in millions of homes and commercial applications across the globe. Our innovative products powered by the myQ digital ecosystem provide cust
      label     score
1349      0 -1.022965
939       0 -1.101435
518       0 -1.197905
1326      0 -1.586607
442       0 -1.633786
3022      0 -1.856516
1833      0 -2.609137
3684      1 -2.733324
Job Description:
 About Hallgate Management: Hallgate Management is a property management company with a strong commitment to providing exceptional service to our clients and residents. We pride ourselves on our dedication to excellence, integrity, and continuous growth.
Position Overview: We are seeking a detail-ori
      label     score
401       0 -1.319430
300       0 -2.293296
1309      0 -2.537764
1129      0 -2.608291
894       0 -3.080945
Job Description:
 About Us Skadd

In [21]:
test_pairs=list(zip(test_df['job_description_text'],test_df['resume_text']))

scores=cross_encoder_model.predict(test_pairs,batch_size=64,show_progress_bar=False)

metrics=model_evaluation(scores,test_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])


Spearman: 0.28843747165346373
Top-3 Accuracy: 0.8928571428571429
NDCG: 0.6309102011399077
MAP: 0.7292615661999167
MRR: 0.848224043715847


In [22]:
results_dir=config.RESULTS_DIR
os.makedirs(results_dir,exist_ok=True)
with open(os.path.join(results_dir,'cross_encoder_finetuned.json'),'w') as f:
    json.dump(metrics,f)